# Introduction to Data Science 2026

# Week 4

In this week's exercise, we look at prompting and zero- and few-shot task settings. Below is a text generation example from https://github.com/TurkuNLP/intro-to-nlp/blob/master/text_generation_pipeline_example.ipynb demonstrating how to load a text generation pipeline with a pre-trained model and generate text with a given prompt. Your task is to load a similar pre-trained generative model and assess whether the model succeeds at a set of tasks in zero-shot, one-shot, and two-shot settings.

**Note: Downloading and running the pre-trained model locally may take some time. Alternatively, you can open and run this notebook on [Google Colab](https://colab.research.google.com/), as assumed in the following example.**

## Text generation example

This is a brief example of how to run text generation with a causal language model and `pipeline`.

Install [transformers](https://huggingface.co/docs/transformers/index) python package. This will be used to load the model and tokenizer and to run generation.

In [19]:
!pip install --quiet transformers

Import the `AutoTokenizer`, `AutoModelForCausalLM`, and `pipeline` classes. The first two support loading tokenizers and generative models from the [Hugging Face repository](https://huggingface.co/models), and the last wraps a tokenizer and a model for convenience.

In [20]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

Load a generative model and its tokenizer. You can substitute any other generative model name here (e.g. [other TurkuNLP GPT-3 models](https://huggingface.co/models?sort=downloads&search=turkunlp%2Fgpt3)), but note that Colab may have issues running larger models. 

In [21]:
MODEL_NAME = 'TurkuNLP/gpt3-finnish-large'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

Instantiate a text generation pipeline using the tokenizer and model.

In [22]:
pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    device=model.device
)

We can now call the pipeline with a text prompt; it will take care of tokenizing, encoding, generation, and decoding:

In [23]:
output = pipe('Terve, miten menee?', max_new_tokens=25)

print(output)

[{'generated_text': 'Terve, miten menee?”\n”Hyvin, kiitos.”\n”Kiva kuulla.”\n”Kuule, minulla on sinulle asiaa.”\n'}]


Just print the text

In [24]:
print(output[0]['generated_text'])

Terve, miten menee?”
”Hyvin, kiitos.”
”Kiva kuulla.”
”Kuule, minulla on sinulle asiaa.”



We can also call the pipeline with any arguments that the model `generate` function supports. For details on text generation using `transformers`, see e.g. [this tutorial](https://huggingface.co/blog/how-to-generate).

Example with sampling and a high `temperature` parameter to generate more chaotic output:

In [26]:
output = pipe(
    'Terve, miten menee?',
    do_sample=True,
    temperature=10.0,
    max_new_tokens=25
)

print(output[0]['generated_text'])

Terve, miten menee? kysyi Heikki yhtäkkiä astuessaan taloon sisään kantaen kahvipöytä-tarvikkeita käsissään sisään tultuaan.
(Ryökäle luuli varmasti hänen tulevan meille istumaan)




## Exercise 1

Your task is to assess whether a generative model succeeds in the following tasks in zero-shot, one-shot, and two-shot settings:

- binary sentiment classification (positive / negative)

- person name recognition

- two-digit addition (e.g. 11 + 22 = 33)

For example, for assessing whether a generative model can name capital cities, we could use the following prompts:

- zero-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""
- one-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Sweden?\
	>Answer: Stockholm
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""
- two-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Sweden?\
	>Answer: Stockholm
	>
	>Question: What is the capital of Denmark?\
	>Answer: Copenhagen
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""

You can do the tasks either in English or Finnish and use a generative model of your choice from the Hugging Face models repository, for example the following models:

- English: `gpt2-large`
- Finnish: `TurkuNLP/gpt3-finnish-large`

You can either come up with your own instructions for the tasks or use the following:

- English:
	- binary sentiment classification: "Do the following texts express a positive or negative sentiment?"
	- person name recognition: "List the person names occurring in the following texts."
	- two-digit addition: "This is a first grade math exam."
- Finnish:
	- binary sentiment classification: "Ilmaisevatko seuraavat tekstit positiivista vai negatiivista tunnetta?"
	- person name recognition: "Listaa seuraavissa teksteissä mainitut henkilönnimet."
	- two-digit addition: "Tämä on ensimmäisen luokan matematiikan koe."

Come up with at least two test cases for each of the three tasks, and come up with your own one- and two-shot examples.

In [1]:

!pip install --quiet transformers

import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

MODEL_NAME = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

tokenizer.pad_token = tokenizer.eos_token

device = 0 if torch.cuda.is_available() else -1
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device
)


def make_prompt(instruction, examples, test_input, input_label):
    """Create a zero-shot, one-shot, or two-shot prompt."""
    prompt = instruction + "\n\n"

    for example_input, example_answer in examples:
        prompt += f"{input_label}: {example_input}\n"
        prompt += f"Answer: {example_answer}\n\n"

    prompt += f"{input_label}: {test_input}\n"
    prompt += "Answer:"
    return prompt


def generate_answer(prompt, max_new_tokens):
    """Generate only the continuation after 'Answer:' and keep its first line."""
    output = pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        return_full_text=False
    )
    return output[0]["generated_text"].split("\n")[0].strip()


def sentiment_is_correct(answer, expected):
    """Check whether the first generated sentiment label is the expected one."""
    match = re.search(r"\b(positive|negative)\b", answer.lower())
    return match is not None and match.group(1) == expected


def names_are_correct(answer, expected_names):
    """Compare capitalized name tokens in the generated first line with the expected names."""
    predicted_names = re.findall(r"\b[A-Z][a-z]+\b", answer)
    return set(predicted_names) == set(expected_names)


def addition_is_correct(answer, expected):
    """Check whether the first integer generated by the model is the expected sum."""
    match = re.search(r"-?\d+", answer)
    return match is not None and int(match.group()) == expected


# -----------------------------------------------------------------------------
# Task 1: Binary sentiment classification
# -----------------------------------------------------------------------------
sentiment_instruction = (
    "Do the following texts express a positive or negative sentiment? "
    "Answer with only positive or negative."
)

sentiment_examples = [
    ("I absolutely loved the movie.", "positive"),
    ("The food was terrible and I will not return.", "negative")
]

sentiment_tests = [
    ("The concert was fantastic and made me very happy.", "positive"),
    ("I regret buying this phone; it is awful.", "negative")
]


# -----------------------------------------------------------------------------
# Task 2: Person name recognition
# -----------------------------------------------------------------------------
name_instruction = (
    "List the person names occurring in the following texts. "
    "Give only the names, separated by commas."
)

name_examples = [
    ("Anna called Mark after work.", "Anna, Mark"),
    ("David had lunch with Sophia on Monday.", "David, Sophia")
]

name_tests = [
    ("Emma met Daniel at the library.", ["Emma", "Daniel"]),
    ("Yesterday Olivia sent a message to Lucas.", ["Olivia", "Lucas"])
]


# -----------------------------------------------------------------------------
# Task 3: Two-digit addition
# -----------------------------------------------------------------------------
addition_instruction = (
    "This is a first grade math exam. "
    "Calculate the sum and answer with only the number."
)

addition_examples = [
    ("23 + 41 =", "64"),
    ("58 + 17 =", "75")
]

addition_tests = [
    ("34 + 25 =", 59),
    ("47 + 36 =", 83)
]


shot_settings = {
    "zero-shot": 0,
    "one-shot": 1,
    "two-shot": 2
}


# tests
def run_task(task_name, instruction, examples, tests, input_label, max_new_tokens, checker):
    print("=" * 80)
    print(task_name)
    print("=" * 80)

    for setting_name, number_of_examples in shot_settings.items():
        print(f"\n{setting_name.upper()}")
        print("-" * 80)

        prompt_examples = examples[:number_of_examples]

        for test_number, (test_input, expected) in enumerate(tests, start=1):
            prompt = make_prompt(
                instruction=instruction,
                examples=prompt_examples,
                test_input=test_input,
                input_label=input_label
            )

            answer = generate_answer(prompt, max_new_tokens=max_new_tokens)
            correct = checker(answer, expected)

            print(f"Test {test_number}")
            print(f"Input:    {test_input}")
            print(f"Expected: {expected}")
            print(f"Model:    {answer}")
            print(f"Correct:  {correct}")
            print()


run_task(
    task_name="Task 1: Binary sentiment classification",
    instruction=sentiment_instruction,
    examples=sentiment_examples,
    tests=sentiment_tests,
    input_label="Text",
    max_new_tokens=5,
    checker=sentiment_is_correct
)

run_task(
    task_name="Task 2: Person name recognition",
    instruction=name_instruction,
    examples=name_examples,
    tests=name_tests,
    input_label="Text",
    max_new_tokens=12,
    checker=names_are_correct
)

run_task(
    task_name="Task 3: Two-digit addition",
    instruction=addition_instruction,
    examples=addition_examples,
    tests=addition_tests,
    input_label="Question",
    max_new_tokens=6,
    checker=addition_is_correct
)



[notice] A new release of pip is available: 24.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


2026-09-27 19:51:34.984422: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-27 19:51:34.986401: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-27 19:51:34.989299: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-27 19:51:34.998759: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-27 19:51:35.017407: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registe

Task 1: Binary sentiment classification

ZERO-SHOT
--------------------------------------------------------------------------------
Test 1
Input:    The concert was fantastic and made me very happy.
Expected: positive
Model:    I was very happy.
Correct:  False

Test 2
Input:    I regret buying this phone; it is awful.
Expected: negative
Model:    I am sorry.
Correct:  False


ONE-SHOT
--------------------------------------------------------------------------------
Test 1
Input:    The concert was fantastic and made me very happy.
Expected: positive
Model:    negative
Correct:  False

Test 2
Input:    I regret buying this phone; it is awful.
Expected: negative
Model:    negative
Correct:  True


TWO-SHOT
--------------------------------------------------------------------------------
Test 1
Input:    The concert was fantastic and made me very happy.
Expected: positive
Model:    positive
Correct:  True

Test 2
Input:    I regret buying this phone; it is awful.
Expected: negative
Model: 

# conclusion:
# The gpt-2 model performed poorly in the zero-shot setting.
# Few-shot examples improved person name recognition, especially in the two-shot setting.
# The improvement in sentiment classification was limited.
# The model failed all two-digit addition test cases in all three settings.

**Submit this exercise by submitting your code and your answers to the above questions as comments on the MOOC platform. You can return this Jupyter notebook (.ipynb) or .py, .R, etc depending on your programming preferences.**